# QTI MLP thesis recreation: train, predict, evaluate

This notebook recreates the supervised `QTI_MLP` training workflow without editing the original Python scripts. All paths and switches are declared at the top. The default run is a smoke test (`1` fold, `1` epoch); set `SMOKE_MODE = False` for the full `18` fold thesis-style recreation.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import sys
import time

PROJECT_ROOT = Path(r"C:\qti_mlp_baseline")
DATA_ROOT = Path(r"B:\ML_TS")
OUTPUT_ROOT = PROJECT_ROOT / "thesis_recreation_outputs"
MODEL_ROOT = OUTPUT_ROOT / "models"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
FIGURE_ROOT = OUTPUT_ROOT / "figures"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
TENSORBOARD_ROOT = PROJECT_ROOT / "runs" / "thesis_recreation"

# Default is deliberately small. Set False for the full 18-fold, 100-epoch recreation.
SMOKE_MODE = True
OVERWRITE_OUTPUTS = False

PATIENT_ORDER = [
    "P1", "P10", "P2", "P11", "P3", "P12", "P4", "P13", "P5",
    "P14", "P6", "P15", "P7", "P16", "P8", "P17", "P9", "P18",
]

NII_NAME = "og_NII_sub_min_dn_db_dg_tp_mc_b0_avg.nii.gz"
MASK_NAME = "manual_mask.nii.gz"
LABEL_REL = Path("qtip_sub_min_pipe_2") / "dps3_nan_clmpd_clmpd.mat"

INVAR_KEYS = ["MD", "FA", "uFA", "C_c", "C_MD"]
IDENTIFIER = "QTI_MLP_16P_sub_min"
OUT_DIR = "Paper_Base_Run_recreated"

LEARNING_RATE = 1e-3
BATCH_SIZE = 512
EPOCHS = 100
FINAL_SIGMOID = False
ZSCORE_OUTPUT = True
NORM_KEYS = INVAR_KEYS if ZSCORE_OUTPUT else ["None"]
SCHEDULE = True
DECAY_CONST = 0.925
SCHED_START = 14
SCHED_END = 74
LAYER_NORM = False
BIAS = True
DEVICE = "cpu"

TS_SIZE = 1
VS_SIZE = 1
ENS_FACTOR = len(PATIENT_ORDER)
SLICE_IND = [[2, 21] for _ in PATIENT_ORDER]

RUN_EPOCHS = 1 if SMOKE_MODE else EPOCHS
FOLDS_TO_RUN = [0] if SMOKE_MODE else list(range(ENS_FACTOR))

for directory in [OUTPUT_ROOT, MODEL_ROOT, PREDICTION_ROOT, FIGURE_ROOT, METRICS_ROOT, TENSORBOARD_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root:    {DATA_ROOT}")
print(f"Output root:  {OUTPUT_ROOT}")
print(f"Mode:         {'SMOKE' if SMOKE_MODE else 'FULL'}")
print(f"Folds:        {FOLDS_TO_RUN}")
print(f"Epochs:       {RUN_EPOCHS}")


In [ ]:
# Environment check. If this fails, create/activate the environment from environment.yml.
import importlib.util

required_modules = [
    "torch", "nibabel", "numpy", "pandas", "scipy", "matplotlib", "tensorboard"
]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(
        "Missing dependencies: " + ", ".join(missing) +
        ". In a terminal from C:\\qti_mlp_baseline, run setup_env.ps1 or install requirements.txt."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Environment looks ready.")


In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
from torch import nn
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import nibabel as nib
import matplotlib.pyplot as plt

from QTI_MLP import QTI_Dataset, QTI_MLP, train_loop, test_loop, rev_zscore

pd.set_option("display.max_rows", 100)


In [ ]:
def build_patient_table(patient_order=PATIENT_ORDER):
    rows = []
    for patient in patient_order:
        base = DATA_ROOT / patient
        rows.append({
            "patient": patient,
            "nii": base / NII_NAME,
            "mask": base / MASK_NAME,
            "label": base / LABEL_REL,
        })
    return pd.DataFrame(rows)

patient_table = build_patient_table()
validation_rows = []
for _, row in patient_table.iterrows():
    validation_rows.append({
        "patient": row["patient"],
        "nii_exists": row["nii"].exists(),
        "mask_exists": row["mask"].exists(),
        "label_exists": row["label"].exists(),
        "nii": str(row["nii"]),
        "mask": str(row["mask"]),
        "label": str(row["label"]),
    })
validation = pd.DataFrame(validation_rows)
display(validation)

missing = validation.loc[~(validation["nii_exists"] & validation["mask_exists"] & validation["label_exists"])]
if len(missing):
    raise FileNotFoundError("Missing required files for patients: " + ", ".join(missing["patient"].tolist()))

print(f"Validated {len(validation)} patients.")


In [ ]:
def make_fold(current_step):
    current_ind = np.arange(len(PATIENT_ORDER))
    current_ind = np.roll(current_ind, shift=-current_step * TS_SIZE)
    ts_ind = current_ind[:TS_SIZE].tolist()
    vs_ind = current_ind[TS_SIZE:TS_SIZE + VS_SIZE].tolist()
    tr_ind = current_ind[TS_SIZE + VS_SIZE:].tolist()
    return {
        "fold": current_step,
        "test_idx": ts_ind,
        "val_idx": vs_ind,
        "train_idx": tr_ind,
        "test_patients": [PATIENT_ORDER[i] for i in ts_ind],
        "val_patients": [PATIENT_ORDER[i] for i in vs_ind],
        "train_patients": [PATIENT_ORDER[i] for i in tr_ind],
    }

folds = pd.DataFrame([
    {
        "fold": f["fold"],
        "test": ",".join(f["test_patients"]),
        "validation": ",".join(f["val_patients"]),
        "n_train": len(f["train_patients"]),
        "train": ",".join(f["train_patients"]),
    }
    for f in [make_fold(step) for step in range(ENS_FACTOR)]
])
display(folds)
assert all(folds["n_train"] == 16), "Expected 16 training patients in every fold."


In [ ]:
def paths_for(indices, column):
    return [str(patient_table.iloc[i][column]) for i in indices]

def slices_for(indices):
    return [SLICE_IND[i] for i in indices]

def prepare_train_or_val_dataset(indices):
    ds = QTI_Dataset(
        paths_for(indices, "nii"),
        paths_for(indices, "label"),
        paths_for(indices, "mask"),
        slices_for(indices),
        INVAR_KEYS,
        zscore_output=ZSCORE_OUTPUT,
    )
    ds.thresh_neg_vals()
    ds.mask_zero_vals()
    ds.mask_nan_vals()
    ds.apply_masked_tensor()
    ds.zscore_norm_input()
    ds.zscore_norm_labels(NORM_KEYS)
    ds.flatten_slice_dim()
    ds.apply_mask()
    return ds

def make_model_name(fold, epoch_number, timestamp):
    ts_names = "".join(fold["test_patients"])
    vs_names = "".join(fold["val_patients"])
    name = (
        f"{IDENTIFIER}_ens_f{ENS_FACTOR}_ts_{ts_names}_vs_{vs_names}_"
        f"{timestamp[:11]}h_ep{epoch_number}_LR{LEARNING_RATE:.0e}_bs{BATCH_SIZE}"
    )
    if FINAL_SIGMOID:
        name += "_sigm"
    elif ZSCORE_OUTPUT:
        name += "_outnorm"
    if LAYER_NORM:
        name += "_lnorm"
    if not BIAS:
        name += "_nobias"
    if SCHEDULE:
        name += f"_sched_{int(100 * DECAY_CONST)}"
    return name


In [ ]:
def train_one_fold(current_step, epochs=RUN_EPOCHS):
    fold = make_fold(current_step)
    print(f"Training fold {current_step}: test={fold['test_patients']} validation={fold['val_patients']}")

    ds_train = prepare_train_or_val_dataset(fold["train_idx"])
    ds_val = prepare_train_or_val_dataset(fold["val_idx"])

    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
    val_loader = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)

    n_scans = ds_train.X.size()[-1]
    n_invars = ds_train.y.size()[-1]
    model = QTI_MLP(n_scans, n_invars, LAYER_NORM, FINAL_SIGMOID, BIAS).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=DECAY_CONST) if SCHEDULE else None
    loss_fn = nn.MSELoss()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    run_name = (
        f"{IDENTIFIER}_ens_f{ENS_FACTOR}_ts_{''.join(fold['test_patients'])}_"
        f"vs_{''.join(fold['val_patients'])}_{timestamp[:11]}h_LR{LEARNING_RATE:.0e}_bs{BATCH_SIZE}"
    )
    writer = SummaryWriter(str(TENSORBOARD_ROOT / OUT_DIR / run_name))

    history = []
    tic = time.time()
    for epoch in range(epochs):
        model.to(DEVICE)
        train_loss = train_loop(train_loader, model, loss_fn, optimizer, DEVICE, BATCH_SIZE)
        val_loss = test_loop(val_loader, model, loss_fn, DEVICE, BATCH_SIZE)

        if SCHEDULE and (SCHED_START <= epoch <= SCHED_END):
            scheduler.step()
        lr = optimizer.param_groups[0]["lr"]

        writer.add_scalars("Training vs. Validation Loss", {"Training": train_loss, "Validation": val_loss}, epoch + 1)
        if SCHEDULE:
            writer.add_scalars("Learning Rate", {"LR": lr}, epoch + 1)
        writer.flush()

        history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss, "lr": lr})
        print(f"epoch {epoch + 1:03d}/{epochs}: train={train_loss:.6g} val={val_loss:.6g} lr={lr:.3g}")

    writer.close()
    elapsed_s = time.time() - tic

    model_dir = MODEL_ROOT / OUT_DIR
    model_dir.mkdir(parents=True, exist_ok=True)
    model_name = make_model_name(fold, epochs, timestamp)
    model_path = model_dir / model_name
    if model_path.exists() and not OVERWRITE_OUTPUTS:
        raise FileExistsError(f"Model already exists: {model_path}")
    torch.save(model.state_dict(), model_path)

    zscore_path = Path(str(model_path) + "_zscore.csv")
    pd.DataFrame(ds_train.zscore.numpy(), index=["mean", "std"]).to_csv(zscore_path, index=False)

    history_path = Path(str(model_path) + "_history.csv")
    pd.DataFrame(history).to_csv(history_path, index=False)

    result = {
        "fold": current_step,
        "test_patients": fold["test_patients"],
        "val_patients": fold["val_patients"],
        "train_patients": fold["train_patients"],
        "model_path": str(model_path),
        "zscore_path": str(zscore_path),
        "history_path": str(history_path),
        "elapsed_s": elapsed_s,
        "epochs": epochs,
    }
    print(json.dumps(result, indent=2))
    return result


In [ ]:
training_results = []
for fold_id in FOLDS_TO_RUN:
    training_results.append(train_one_fold(fold_id, epochs=RUN_EPOCHS))

manifest_path = METRICS_ROOT / ("smoke_training_manifest.csv" if SMOKE_MODE else "training_manifest.csv")
pd.DataFrame(training_results).to_csv(manifest_path, index=False)
display(pd.DataFrame(training_results))
print(f"Wrote manifest: {manifest_path}")


In [ ]:
def prepare_prediction_dataset(patient_index):
    ds = QTI_Dataset(
        [str(patient_table.iloc[patient_index]["nii"])],
        scalar_invars_path=None,
        mask_path=[str(patient_table.iloc[patient_index]["mask"])],
        slice_ind=[SLICE_IND[patient_index]],
        invar_keys=INVAR_KEYS,
        zscore_output=ZSCORE_OUTPUT,
    )
    ds.thresh_neg_vals()
    ds.apply_masked_tensor()
    ds.zscore_norm_input()
    output_shape = [ds.X.size(i) for i in range(len(ds.X.size()) - 1)]
    ds.flatten_slice_dim()
    return ds, output_shape

def predict_one_fold(train_result):
    fold = make_fold(int(train_result["fold"]))
    patient_index = fold["test_idx"][0]
    patient = fold["test_patients"][0]

    ds_pred, output_shape = prepare_prediction_dataset(patient_index)
    n_scans = ds_pred.X.size()[-1]
    model = QTI_MLP(n_scans, len(INVAR_KEYS), LAYER_NORM, FINAL_SIGMOID, BIAS)
    state = torch.load(train_result["model_path"], map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()

    with torch.no_grad():
        pred = model(ds_pred.X.to(DEVICE))

    if ZSCORE_OUTPUT:
        zs = torch.tensor(pd.read_csv(train_result["zscore_path"]).values.astype(float), dtype=torch.float32)
        pred = rev_zscore(pred.cpu(), zs)
    pred = torch.nan_to_num(pred, nan=0.0)

    output_shape.append(pred.size(-1))
    pred = torch.permute(torch.reshape(pred, output_shape), (0, 2, 3, 1, 4)).detach().cpu().numpy()

    out_dir = PREDICTION_ROOT / OUT_DIR / patient
    out_dir.mkdir(parents=True, exist_ok=True)
    identifier = Path(train_result["model_path"]).name
    saved = []
    for ind, invar in enumerate(INVAR_KEYS):
        out_file = out_dir / f"{identifier}_{invar}_prediction.nii.gz"
        if out_file.exists() and not OVERWRITE_OUTPUTS:
            raise FileExistsError(f"Prediction already exists: {out_file}")
        img = nib.Nifti1Image(pred[0, ..., ind], ds_pred.nii_affines[0])
        nib.save(img, str(out_file))
        saved.append(str(out_file))
    return {"fold": train_result["fold"], "patient": patient, "prediction_dir": str(out_dir), "prediction_files": saved}


In [ ]:
prediction_results = [predict_one_fold(result) for result in training_results]
pred_manifest_path = METRICS_ROOT / ("smoke_prediction_manifest.csv" if SMOKE_MODE else "prediction_manifest.csv")
pd.DataFrame(prediction_results).to_csv(pred_manifest_path, index=False)
display(pd.DataFrame(prediction_results))
print(f"Wrote prediction manifest: {pred_manifest_path}")


In [ ]:
def load_target_volume(patient, invar):
    mat_path = DATA_ROOT / patient / LABEL_REL
    return sio.loadmat(mat_path)["dps"][invar][0, 0].astype(np.float32)

def load_mask(patient, patient_index):
    mask = nib.load(str(DATA_ROOT / patient / MASK_NAME)).get_fdata().astype(bool)
    first, last = SLICE_IND[patient_index]
    mask[:, :, :first] = False
    mask[:, :, last + 1:] = False
    return mask

def compute_prediction_metrics(prediction_result):
    patient = prediction_result["patient"]
    patient_index = PATIENT_ORDER.index(patient)
    mask = load_mask(patient, patient_index)
    rows = []
    pred_files = {Path(p).name.split("_prediction.nii.gz")[0].split("_")[-1]: p for p in prediction_result["prediction_files"]}

    # The split above is fragile for C_MD, so map by suffix explicitly.
    pred_files = {}
    for p in prediction_result["prediction_files"]:
        path = Path(p)
        for invar in INVAR_KEYS:
            if path.name.endswith(f"_{invar}_prediction.nii.gz"):
                pred_files[invar] = path

    for invar in INVAR_KEYS:
        target = load_target_volume(patient, invar)
        pred = nib.load(str(pred_files[invar])).get_fdata().astype(np.float32)
        valid = mask & np.isfinite(target) & np.isfinite(pred)
        diff = pred[valid] - target[valid]
        rmse = float(np.sqrt(np.mean(diff ** 2)))
        target_range = float(np.nanmax(target[valid]) - np.nanmin(target[valid]))
        rows.append({
            "fold": prediction_result["fold"],
            "patient": patient,
            "invar": invar,
            "n_voxels": int(np.count_nonzero(valid)),
            "mae": float(np.mean(np.abs(diff))),
            "mse": float(np.mean(diff ** 2)),
            "rmse": rmse,
            "nrmse_range": rmse / target_range if target_range > 0 else np.nan,
            "bias": float(np.mean(diff)),
        })
    return rows

metric_rows = []
for pred_result in prediction_results:
    metric_rows.extend(compute_prediction_metrics(pred_result))

metrics = pd.DataFrame(metric_rows)
metrics_path = METRICS_ROOT / ("smoke_metrics.csv" if SMOKE_MODE else "metrics.csv")
metrics.to_csv(metrics_path, index=False)
display(metrics)
print(f"Wrote metrics: {metrics_path}")


In [ ]:
def plot_loss_curve(history_path):
    history_path = Path(history_path)
    history = pd.read_csv(history_path)
    fig, ax = plt.subplots(figsize=(7, 4), dpi=150)
    ax.plot(history["epoch"], history["train_loss"], label="training", linewidth=2)
    ax.plot(history["epoch"], history["val_loss"], label="validation", linewidth=2)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss / voxel")
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    if "lr" in history:
        ax2 = ax.twinx()
        ax2.plot(history["epoch"], history["lr"], color="0.4", linestyle="--", label="lr")
        ax2.set_ylabel("learning rate")
    fig.tight_layout()
    out = FIGURE_ROOT / f"{history_path.stem}_loss.png"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out

def plot_prediction_panel(prediction_result, slice_index=None):
    patient = prediction_result["patient"]
    patient_index = PATIENT_ORDER.index(patient)
    if slice_index is None:
        mask = load_mask(patient, patient_index)
        z_counts = mask.sum(axis=(0, 1))
        slice_index = int(np.argmax(z_counts))

    pred_files = {}
    for p in prediction_result["prediction_files"]:
        path = Path(p)
        for invar in INVAR_KEYS:
            if path.name.endswith(f"_{invar}_prediction.nii.gz"):
                pred_files[invar] = path

    fig, axes = plt.subplots(3, len(INVAR_KEYS), figsize=(3.0 * len(INVAR_KEYS), 8.0), dpi=150)
    for c, invar in enumerate(INVAR_KEYS):
        target = load_target_volume(patient, invar)[:, :, slice_index]
        pred = nib.load(str(pred_files[invar])).get_fdata()[:, :, slice_index]
        diff = pred - target
        panels = [(target, "target"), (pred, "prediction"), (np.abs(diff), "abs diff")]
        for r, (img, title) in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(np.rot90(img), cmap="viridis")
            ax.set_title(f"{invar} {title}")
            ax.axis("off")
    fig.suptitle(f"{patient}, slice {slice_index}")
    fig.tight_layout()
    out = FIGURE_ROOT / f"{patient}_prediction_panel_slice{slice_index}.png"
    fig.savefig(out, bbox_inches="tight")
    plt.show()
    return out

loss_figures = [plot_loss_curve(result["history_path"]) for result in training_results]
panel_figures = [plot_prediction_panel(result) for result in prediction_results]
print("Loss figures:", loss_figures)
print("Prediction panels:", panel_figures)


In [ ]:
# Optional helper: summarize a full run after SMOKE_MODE=False.
summary = None
if metrics_path.exists():
    summary = metrics.groupby("invar")[["mae", "mse", "rmse", "nrmse_range", "bias"]].agg(["mean", "std"])
    summary_path = METRICS_ROOT / ("smoke_metrics_summary.csv" if SMOKE_MODE else "metrics_summary.csv")
    summary.to_csv(summary_path)
    display(summary)
    print(f"Wrote summary: {summary_path}")
